<a href="https://colab.research.google.com/github/Harsh-Prajapati54/LLMs---Playbook/blob/main/Document_Loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # Document Loader

 in this note book i am going to implement pipline for injection and preprocessing of `pdf` using `PyMupdf `

***Script that ingests a PDF, extracts text, splits into question-level chunks, and saves as structured JSON — zero libraries except PyMuPDF.***

 ### setup

In [1]:
%%capture
!pip install pymupdf

In [2]:
import pymupdf
print(pymupdf.__doc__)

PyMuPDF 1.27.2.2: Python bindings for the MuPDF 1.27.2 library.
Python 3.12 running on linux (64-bit).



In [3]:
%%capture
!pip install -U langchain-text-splitters

In [40]:
%%capture
!pip install langchain-community

## scripts for opening and preprocessing the pdf document

In [4]:
# loads the document
doc = pymupdf.open("/content/Hello transformers.pdf")

In [5]:
# count the pages in document
pages_in_doc = doc.page_count
pages_in_doc

55

Extracting text from Pdf

In [20]:
doc = pymupdf.open("/content/Hello transformers.pdf")
# cretating a text output
out = open("extracted_text.txt","wb")
# iterating over pages
for page in doc:
  tp = page.get_textpage_ocr()
  text = page.get_text(textpage = tp).encode("utf8") # get plain text (is in UTF-8)
  out.write(text) # write text of pages
  out.write(f"\n--- Page {page.number + 1} ---\n".encode("utf8"))

out.close()

In [18]:
with open("extracted_text.txt","r") as f:
    full_document_content = f.read()
# The `extracted_doc` variable (file object) is no longer needed after reading its content into `full_document_content`.

## Chunking the text

In [30]:
import pymupdf
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 512, chunk_overlap = 100)
texts = text_splitter.split_text(full_document_content)


print(f"Total chunks: {len(texts)}")
print(texts[271])



Total chunks: 273
Chapter 9, we’ll explore some techniques to deal with this situation.
Now that we’ve seen what’s involved in training and sharing a transformer, in the next
chapter we’ll explore implementing our very own transformer model from scratch.
Conclusion 
| 
55


## Adding Metadata

In [37]:
from datetime import date
from langchain_core.documents import Document

# ── fill these for your document ──────────────────────────────────────
SOURCE_PATH  = "your_file.pdf"
DOC_TYPE     = "generic"          # e.g. "report", "invoice", "manual"
DOC_TITLE    = "Your Document Title"
DOC_AUTHOR   = "Author Name"
CREATION_DATE = "2024-01-01"       # ISO format from pdf.metadata["creationDate"]
TOTAL_PAGES  = 1                  # from pymupdf doc.page_count

total  = len(texts)
today  = date.today().isoformat()

# wrap each plain string → Document with full metadata
docs = []
for idx, chunk_text in enumerate(texts):
    meta = {
        # document-level (same for every chunk)
        "source":         SOURCE_PATH,
        "title":          DOC_TITLE,
        "author":         DOC_AUTHOR,
        "creation_date":  CREATION_DATE,
        "total_pages":    TOTAL_PAGES,
        "doc_type":       DOC_TYPE,

        # chunk-level (unique per chunk)
        "chunk_index":    idx,
        "total_chunks":   total,
        "char_count":     len(chunk_text),
        "prev_chunk_id":  idx - 1 if idx > 0          else None,
        "next_chunk_id":  idx + 1 if idx < total - 1  else None,
        "ingestion_date": today,
    }
    docs.append(Document(page_content=chunk_text, metadata=meta))

# verify
print(f"Total documents: {len(docs)}")
print(f"\nSample metadata (chunk 271):")
for k, v in docs[271].metadata.items():
    print(f"  {k:<18} {v}")

Total documents: 273

Sample metadata (chunk 271):
  source             your_file.pdf
  title              Your Document Title
  author             Author Name
  creation_date      2024-01-01
  total_pages        1
  doc_type           generic
  chunk_index        271
  total_chunks       273
  char_count         254
  prev_chunk_id      270
  next_chunk_id      272
  ingestion_date     2026-04-09


alternate cell that adds metadata and splits data(text) -> chunks

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Step 1 — load: each page becomes a Document with metadata auto-populated
loader = PyMuPDFLoader("/content/Hello transformers.pdf")
pages  = loader.load()

print(f"Pages loaded: {len(pages)}")
print(f"Sample page metadata:\n", pages[0].metadata)

# Step 2 — split: metadata is automatically copied into every chunk
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 512,
    chunk_overlap = 100,
)
docs = text_splitter.split_documents(pages)  # NOT split_text()

print(f"\nTotal chunks: {len(docs)}")
print(f"\nChunk 271 metadata:\n", docs[271].metadata)
print(f"\nChunk 271 content:\n",  docs[271].page_content)